# 🔬 ICT YOLOv8 Pro Fine-Tuner: Institutional Quality Optimization

### 🎯 4 Advanced Fine-Tuning Optimizations:
1. **Unmitigated (Fresh) Zones Only:** Historical filled/broken gaps are filtered out. Model learns to highlight ONLY live actionable zones.
2. **Exact TradingView Dark Theme Match:** Charts styled with TradingView's `#131722` dark canvas, `#089981` green, and `#F23645` red.
3. **No-Mosaic Time-Series Training (`mosaic=0.0`, `flipud=0.0`):** Preserves true sequential price action.
4. **Transfer Learning Fine-Tuning:** Loads existing trained weights (`best.pt` / `best_v2_4class.pt`) and refines them at `imgsz=800` with Cosine LR.

### ⚡ Hardware:
Set Runtime to **T4 GPU** (`Runtime > Change runtime type > T4 GPU > Save`), then click **Runtime > Run all**.

### Step 1: Install Dependencies & Import Libraries

In [ ]:
!pip install -q ultralytics mplfinance huggingface_hub pyarrow

import os, shutil, glob, zoneinfo
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import mplfinance as mpf
from huggingface_hub import hf_hub_download
from ultralytics import YOLO

plt.ioff()
NY_TZ = zoneinfo.ZoneInfo("America/New_York")
print("✅ Fine-Tuner Pro environment ready!")

### Step 2: Load Historical XAU/USD 15m Gold Data

In [ ]:
REPO = "SpatialGhost/xauusd-gold-price-historical-data-2004-2025"
FILE = "XAU_15m_data.jsonl"
START_YEAR = 2023

print("⏳ Fetching 15m Gold dataset...")
path = hf_hub_download(REPO, FILE, repo_type="dataset")

chunks = []
for c in pd.read_json(path, lines=True, chunksize=150000):
    c = c.rename(columns=lambda x: str(x).strip().capitalize())
    c["Date"] = pd.to_datetime(c["Date"], errors="coerce")
    c = c.dropna(subset=["Date"])
    c = c[c["Date"].dt.year >= START_YEAR]
    chunks.append(c)

df_raw = pd.concat(chunks)
if df_raw["Date"].dt.tz is None:
    df_raw["Date"] = df_raw["Date"].dt.tz_localize("UTC")
df_raw["Date_NY"] = df_raw["Date"].dt.tz_convert(NY_TZ)
df = df_raw.set_index("Date_NY").sort_index()
df = df[~df.index.duplicated()][["Open", "High", "Low", "Close", "Volume"]].dropna()

print(f"✅ Clean 15m Data Loaded: {len(df):,} candles!")

### Step 3: Strict Institutional Auto-Labeler (FRESH / UNMITIGATED ZONES ONLY)

In [ ]:
shutil.rmtree("ds_finetune", ignore_errors=True)
for s in ["train", "val"]:
    os.makedirs(f"ds_finetune/images/{s}", exist_ok=True)
    os.makedirs(f"ds_finetune/labels/{s}", exist_ok=True)

W, STEP, DPI, MAX_IMAGES = 100, 45, 100, 1800

# Custom TradingView Color Palette
mc = mpf.make_marketcolors(
    up='#089981', down='#F23645',
    edge={'up': '#089981', 'down': '#F23645'},
    wick={'up': '#089981', 'down': '#F23645'}
)
tv_style = mpf.make_mpf_style(base_mpf_style='nightclouds', marketcolors=mc, facecolor='#131722', edgecolor='#131722', figcolor='#131722')

def find_fresh_zones(w):
    hi, lo, op, cl = w.High.values, w.Low.values, w.Open.values, w.Close.values
    avg_range = (w.High - w.Low).mean()
    min_fvg = avg_range * 0.45
    zones = []
    N = len(w)
    
    for i in range(2, N - 1):
        # 1. Bullish FVG (Class 0)
        if lo[i] - hi[i-2] > min_fvg:
            fvg_top = lo[i]
            fvg_bot = hi[i-2]
            fvg_ce = (fvg_top + fvg_bot) / 2  # 50% Consequent Encroachment
            
            # Mitigation Check: Did subsequent candles violate 50% CE?
            is_mitigated = False
            end_idx = min(i + 14, N - 1)
            for j in range(i + 1, end_idx + 1):
                if cl[j] < fvg_ce:  # Closed below 50% CE -> Mitigated / Broken
                    is_mitigated = True
                    break
            if not is_mitigated:
                zones.append((0, i-2, fvg_bot, fvg_top, end_idx))  # Class 0: Fresh Bull FVG
                
                # Associated Bullish Order Block (Class 2)
                for ob_i in range(i-2, max(0, i-5), -1):
                    if cl[ob_i] < op[ob_i]: # Down candle
                        ob_mt = (op[ob_i] + cl[ob_i]) / 2  # 50% Mean Threshold
                        # Check if OB held
                        if not any(cl[k] < ob_mt for k in range(ob_i+1, min(ob_i+10, N))):
                            zones.append((2, ob_i, lo[ob_i], hi[ob_i], min(ob_i+12, N-1)))
                        break

        # 2. Bearish FVG (Class 1)
        if lo[i-2] - hi[i] > min_fvg:
            fvg_top = lo[i-2]
            fvg_bot = hi[i]
            fvg_ce = (fvg_top + fvg_bot) / 2
            
            # Mitigation Check: Did subsequent candles violate 50% CE?
            is_mitigated = False
            end_idx = min(i + 14, N - 1)
            for j in range(i + 1, end_idx + 1):
                if cl[j] > fvg_ce:  # Closed above 50% CE -> Mitigated / Broken
                    is_mitigated = True
                    break
            if not is_mitigated:
                zones.append((1, i-2, fvg_bot, fvg_top, end_idx))  # Class 1: Fresh Bear FVG
                
                # Associated Bearish Order Block (Class 3)
                for ob_i in range(i-2, max(0, i-5), -1):
                    if cl[ob_i] > op[ob_i]: # Up candle
                        ob_mt = (op[ob_i] + cl[ob_i]) / 2
                        if not any(cl[k] > ob_mt for k in range(ob_i+1, min(ob_i+10, N))):
                            zones.append((3, ob_i, lo[ob_i], hi[ob_i], min(ob_i+12, N-1)))
                        break
    return zones

starts = list(range(0, len(df) - W, STEP))[-MAX_IMAGES:]
split_at = int(len(starts) * 0.8)
saved = 0
print("⏳ Generating High-Definition TradingView Styled Labeled Charts...")

for k, st in enumerate(starts):
    w = df.iloc[st:st+W]
    zones = find_fresh_zones(w)
    if not zones:
        continue
    s = "train" if k < split_at else "val"
    fig, axes = mpf.plot(w, type="candle", style=tv_style, axisoff=True,
                         figsize=(8, 8), returnfig=True)
    fig.set_dpi(DPI)
    ax = axes[0]
    fig.canvas.draw()
    H = 8 * DPI
    lines = []
    for c, i0, zl, zh, i1 in zones:
        x0, y0 = ax.transData.transform((i0 - 0.5, zl))
        x1, y1 = ax.transData.transform((i1 + 0.5, zh))
        xc = (x0 + x1) / 2 / H
        yc = 1.0 - (y0 + y1) / 2 / H
        bw = abs(x1 - x0) / H
        bh = abs(y1 - y0) / H
        xc, yc, bw, bh = max(0, min(1, xc)), max(0, min(1, yc)), max(0.01, min(1, bw)), max(0.01, min(1, bh))
        lines.append(f"{c} {xc:.6f} {yc:.6f} {bw:.6f} {bh:.6f}")
        
    fig.savefig(f"ds_finetune/images/{s}/{k}.png", dpi=DPI, facecolor='#131722')
    plt.close(fig)
    with open(f"ds_finetune/labels/{s}/{k}.txt", "w") as f:
        f.write("\n".join(lines))
    saved += 1
    if saved % 250 == 0:
        print(f"  Progress: {saved} TradingView charts styled & labeled...")

yaml_data = """path: /content/ds_finetune
train: images/train
val: images/val
names:
  0: bull_fvg
  1: bear_fvg
  2: bull_ob
  3: bear_ob
"""
with open("ds_finetune/data.yaml", "w") as f:
    f.write(yaml_data)

print(f"✅ High-Quality Fine-Tuning Dataset Created! Total Images: {saved}")

### Step 4: Transfer Learning Fine-Tuning (Pretrained Weights + No Mosaic)

In [ ]:
import torch
if not torch.cuda.is_available():
    raise RuntimeError("❌ GPU active nahi hai! Runtime > Change runtime type > T4 GPU lagao!")

# Check for existing weights to fine-tune from
base_weights = "yolov8n.pt"
for candidate in ["/content/best_v2_4class.pt", "/content/best.pt", "yolov8n.pt"]:
    if os.path.exists(candidate):
        base_weights = candidate
        break

print(f"⚡ Transfer Learning: Fine-tuning on top of '{base_weights}'!")
print("🚀 Hyperparameters: mosaic=0.0 (Preserves Candle Sequence), imgsz=800, cos_lr=True")

model = YOLO(base_weights)

results = model.train(
    data="/content/ds_finetune/data.yaml",
    epochs=30,
    imgsz=800,              # Crisp wick & micro-gap resolution
    batch=16,
    project="/content/runs",
    name="finetuned_pro",
    exist_ok=True,
    plots=False,
    mosaic=0.0,             # NO MOSAIC: Candlestick time-series order intact
    flipud=0.0,             # NO VERTICAL FLIP: Gravity of candles intact
    fliplr=0.2,             # Minimal horizontal variation
    cos_lr=True,            # Cosine Learning Rate for smooth convergence
    lr0=0.008,
    patience=15
)

best_pro = "/content/runs/finetuned_pro/weights/best.pt"
print("\n🎉 FINE-TUNING COMPLETE! PRO WEIGHTS:", best_pro)

### Step 5: Class-by-Class Validation Metrics Audit

In [ ]:
print("=== 📊 FINE-TUNED PER-CLASS METRICS ===\n")
val_metrics = model.val(imgsz=800)
c_names = ["bull_fvg", "bear_fvg", "bull_ob", "bear_ob"]

print(f"{'Class':<14} {'Precision':<12} {'Recall':<12} {'mAP50':<12}")
print("-" * 50)
for i, cn in enumerate(c_names):
    p = val_metrics.box.p[i] if i < len(val_metrics.box.p) else 0.0
    r = val_metrics.box.r[i] if i < len(val_metrics.box.r) else 0.0
    ap = val_metrics.box.ap50[i] if i < len(val_metrics.box.ap50) else 0.0
    print(f"{cn:<14} {p:<12.3f} {r:<12.3f} {ap:<12.3f}")

print("-" * 50)
print(f"⭐ Overall All-Class mAP50: {val_metrics.box.map50:.3f}")

### Step 6: 1-Click Push to GitHub Repo (No Mobile Download Needed!)

In [ ]:
print("📥 Model Save Options...")
from google.colab import files, drive

# 1. Save to Google Drive
try:
    drive.mount('/content/drive')
    D = '/content/drive/MyDrive/chart_model'
    os.makedirs(D, exist_ok=True)
    shutil.copy('/content/runs/finetuned_pro/weights/best.pt', f'{D}/best_v2_4class.pt')
    print(f"✅ Successfully saved to Google Drive: {D}/best_v2_4class.pt")
except Exception as e:
    print("Drive backup note:", e)

# 2. Direct Download
print("📥 Downloading best_v2_4class.pt...")
try:
    files.download('/content/runs/finetuned_pro/weights/best.pt')
except Exception:
    pass
print("\n🎉 Fine-Tuned Model is ready! You can also find it in Colab sidebar: runs/finetuned_pro/weights/best.pt")